## Fortono to output kai pairno tin lista apo ta unique transcripts gia na kano request meta

In [1]:
import pandas as pd

df = pd.read_csv(
    'outputfull.tsv',
    sep='\t',
    comment='#',
    header=None
)

# Find the header line (starts with single '#')
with open('output.tsv') as f:
    for line in f.readlines():
        if line.startswith('#') and not line.startswith('##'):
            header = line.lstrip('#').strip().split('\t')
            break

df.columns = header # type: ignore

In [2]:
len(df)

49318303

In [5]:
df = df[df['Consequence'].str.contains('3_prime_UTR_variant')]

In [6]:
df

,Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,Existing_variation,Extra
11793,chr1_71298_G/A,chr1:71298,A,ENSG00000186092,ENST00000641515,Transcript,3_prime_UTR_variant,2331,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1
74226,rs56972694,chr1:943527-943529,-,ENSG00000188976,ENST00000870725,Transcript,3_prime_UTR_variant,3483-3485,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1
74292,rs6605067,chr1:944296,A,ENSG00000188976,ENST00000327044,Transcript,3_prime_UTR_variant,2664,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1
74293,rs6605067,chr1:944296,A,ENSG00000187634,ENST00000341065,Transcript,3_prime_UTR_variant,1912,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1;FLAGS=cds_start_NF
74294,rs6605067,chr1:944296,A,ENSG00000187634,ENST00000342066,Transcript,3_prime_UTR_variant,2279,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49287890,chrY_5740126_T/TAC,chrY:5740125-5740126,AC,ENSG00000099715,ENST00000698851,Transcript,3_prime_UTR_variant,6451-6452,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1
49287891,chrY_5740137_T/C/*,chrY:5740137,C,ENSG00000099715,ENST00000400457,Transcript,3_prime_UTR_variant,6952,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1
49287892,chrY_5740137_T/C/*,chrY:5740137,C,ENSG00000099715,ENST00000698851,Transcript,3_prime_UTR_variant,6463,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1
49287893,rs2556915,chrY:5740179,T,ENSG00000099715,ENST00000400457,Transcript,3_prime_UTR_variant,6994,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1


In [4]:
transcripts = df['Feature'].drop_duplicates().to_list()
transcripts

['ENST00000641515',
 'ENST00000870725',
 'ENST00000327044',
 'ENST00000341065',
 'ENST00000342066',
 'ENST00000616016',
 'ENST00000618323',
 'ENST00000870726',
 'ENST00000870727',
 'ENST00000870728',
 'ENST00000870729',
 'ENST00000870730',
 'ENST00000870731',
 'ENST00000870732',
 'ENST00000870733',
 'ENST00000870734',
 'ENST00000870735',
 'ENST00000870736',
 'ENST00000870737',
 'ENST00000870738',
 'ENST00000870739',
 'ENST00000934932',
 'ENST00000934933',
 'ENST00000934934',
 'ENST00000934935',
 'ENST00000934936',
 'ENST00000934937',
 'ENST00000934938',
 'ENST00000934939',
 'ENST00000934940',
 'ENST00000934941',
 'ENST00000934942',
 'ENST00000934943',
 'ENST00000934944',
 'ENST00000934945',
 'ENST00000934946',
 'ENST00000934947',
 'ENST00000934948',
 'ENST00000934949',
 'ENST00000934950',
 'ENST00000934951',
 'ENST00000934952',
 'ENST00000934953',
 'ENST00000934954',
 'ENST00000934955',
 'ENST00000934956',
 'ENST00000934957',
 'ENST00000934958',
 'ENST00000934959',
 'ENST00000968542',


In [18]:
with open('unique_transcripts.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(map(str, transcripts)))

In [ ]:
num_files = 22
chunk_size = len(transcripts) // num_files

for i in range(num_files):
    start_idx = i * chunk_size
    end_idx = start_idx + chunk_size if i < num_files - 1 else len(transcripts)
    
    with open(f'trans{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(map(str, transcripts[start_idx:end_idx])))

In [5]:
import requests

transcriptsNum = 20

df2 = df.head(transcriptsNum)
downloadTranscripts = df2['Feature'].drop_duplicates().to_list()

url = "http://www.ensembl.org/biomart/martservice"
query_xml = f"""<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE Query>
<Query  virtualSchemaName = "default" formatter = "FASTA" header = "0" uniqueRows = "0" count = "" datasetConfigVersion = "0.6" >
			
	<Dataset name = "hsapiens_gene_ensembl" interface = "default" >
		<Filter name = "ensembl_transcript_id" value = "{','.join(downloadTranscripts)}"/>
		<Attribute name = "ensembl_transcript_id" />
		<Attribute name = "cdna" />
	</Dataset>
</Query>"""

res = requests.get(url, params={"query": query_xml})
with open('./TarPmiR_Linux/cDNA.fasta', 'w') as file:
    file.write(res.text)

In [15]:
input_path = './output/miraw_mirna.fasta_miraw_mrna.fasta0.bp'
num_parts = 10

base, ext = input_path.rsplit('.', 1)
part_paths = [f'{base}_part{i+1}.{ext}' for i in range(num_parts)]

with open(input_path, 'w', encoding='utf-8') as f_out:
    for part_path in part_paths:
        with open(part_path, 'r', encoding='utf-8') as f_in:
            f_out.write(f_in.read())


In [14]:
import re
with open('output/miraw_mirna.fasta_miraw_mrna.fasta0_part10.bp', 'r', encoding='utf-8') as f_in:
    content = f_in.read()
content = re.sub(r'(?<=.)hsa', r'\nhsa', content)
content = re.sub(r'(?<=.)MIMAT', r'\nMIMAT', content)
with open('output/miraw_mirna.fasta_miraw_mrna.fasta0_part10.bp', 'w', encoding='utf-8') as f_out:
    f_out.write(content)

In [1]:

import re
from io import StringIO

with open('./TarPmiR_Linux/cDNA.fasta', 'r', encoding='utf-8') as f:
    fasta_ids = [line.strip()[1:] for line in f if line.startswith('>')]
    
with open('./TarPmiR_Linux/mirnatest.fasta_cDNA.fasta.bp', 'r', encoding='utf-8') as f_in:
    content = f_in.read()
content = re.sub(r'(?<=.)hsa', r'\nhsa', content)
with open('./TarPmiR_Linux/mirnatest.fasta_cDNA.fasta.bp', 'w', encoding='utf-8') as f_out:
    f_out.write(content)
dftarp = pd.read_csv(StringIO(content), sep='\t')

dftarp.columns = ['miRNA','mRNA','binding_site','binding_probability','energy','seed','accessibility','AU_content','PhyloP_Stem','PyloP_Flanking','m/e','number_of_pairings','binding_region_length','longest_consecutive_pairings','position_of_longest_consecutive_pairings','pairings_in_3prime_end','difference_of_pairings_between_seed_and_3prime_end']
dftarp.loc[:, 'miRNA'] = dftarp['miRNA'].str.split().str[0]
binds = dftarp.groupby('mRNA')['miRNA'].apply(lambda s: ';'.join(map(str, s))).to_dict()

df2.loc[:, 'hasSequence'] = df2['Feature'].isin(fasta_ids)
# df2.loc[:, 'miRNAHits'] = df2['Feature'].map(binds).where(df2['Feature'].map(binds).notna(), None)

# parse binding_site to numeric start/end
dftarp[['bs_start', 'bs_end']] = (
    dftarp['binding_site'].str.split(',', expand=True).astype(int)
)

df2

NameError: name 'pd' is not defined

In [7]:
dftarp

,miRNA,mRNA,binding_site,binding_probability,energy,seed,accessibility,AU_content,PhyloP_Stem,PyloP_Flanking,m/e,number_of_pairings,binding_region_length,longest_consecutive_pairings,position_of_longest_consecutive_pairings,pairings_in_3prime_end,difference_of_pairings_between_seed_and_3prime_end,bs_start,bs_end
0,hsa-miR-30c-2-3p,ENST00000870726,"2300,2323",0.692308,-21.7,0,4.511229e-09,0.250,0,0,-10.752244,18,23,7,8,8,3,2300,2323
1,hsa-miR-30c-2-3p,ENST00000870726,"814,851",0.153846,-20.2,0,9.222966e-07,0.368,0,0,-8.865139,17,37,7,10,6,0,814,851
2,hsa-miR-30d-5p,ENST00000870726,"2648,2668",0.923077,-19.7,0,1.191108e-07,0.294,0,0,-15.407181,15,20,11,9,6,2,2648,2668
3,hsa-miR-30d-5p,ENST00000870726,"2460,2475",0.461538,-19.1,0,8.027291e-04,0.441,0,0,-17.880193,12,15,10,3,5,4,2460,2475
4,hsa-miR-30d-5p,ENST00000870726,"45,83",0.307692,-16.7,0,3.768705e-08,0.294,0,0,-16.846914,16,38,6,7,7,4,45,83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1320,hsa-miR-218-1-3p,ENST00000870727,"499,522",0.538462,-22.7,0,2.554646e-05,0.441,0,0,-12.318121,17,23,7,16,8,2,499,522
1321,hsa-miR-218-2-3p,ENST00000870727,"2749,2770",1.000000,-29.3,1,3.553495e-05,0.500,0,0,-6.674745,17,21,11,1,6,1,2749,2770
1322,hsa-miR-218-2-3p,ENST00000870727,"1121,1144",1.000000,-25.1,1,1.516049e-04,0.382,0,0,-4.170624,19,23,13,4,5,3,1121,1144
1323,hsa-miR-219a-5p,ENST00000870727,"775,797",0.692308,-15.3,0,1.148641e-03,0.368,0,0,-10.762870,16,22,9,1,5,2,775,797


In [7]:
df2

,Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,Existing_variation,Extra,hasSequence,miRNADisrupt
11793,chr1_71298_G/A,chr1:71298,A,ENSG00000186092,ENST00000641515,Transcript,3_prime_UTR_variant,2331,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1,True,hsa-miR-224-5p;hsa-miR-3126-3p;hsa-miR-5586-3p
74226,rs56972694,chr1:943527-943529,-,ENSG00000188976,ENST00000870725,Transcript,3_prime_UTR_variant,3483-3485,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1,True,None
74292,rs6605067,chr1:944296,A,ENSG00000188976,ENST00000327044,Transcript,3_prime_UTR_variant,2664,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1,True,hsa-miR-205-3p;hsa-miR-6733-5p
74293,rs6605067,chr1:944296,A,ENSG00000187634,ENST00000341065,Transcript,3_prime_UTR_variant,1912,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1;FLAGS=cds_start_NF,True,hsa-miR-152-5p;hsa-miR-301a-5p;hsa-miR-550a-3p...
74294,rs6605067,chr1:944296,A,ENSG00000187634,ENST00000342066,Transcript,3_prime_UTR_variant,2279,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1,True,hsa-miR-152-5p;hsa-miR-550a-3p;hsa-miR-627-3p
74303,rs6605067,chr1:944296,A,ENSG00000187634,ENST00000616016,Transcript,3_prime_UTR_variant,3187,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1,True,hsa-miR-152-5p;hsa-miR-550a-3p;hsa-miR-627-3p
74307,rs6605067,chr1:944296,A,ENSG00000187634,ENST00000618323,Transcript,3_prime_UTR_variant,3190,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1,True,hsa-miR-152-5p;hsa-miR-550a-3p;hsa-miR-627-3p
74310,rs6605067,chr1:944296,A,ENSG00000188976,ENST00000870725,Transcript,3_prime_UTR_variant,2716,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1,True,hsa-miR-205-3p;hsa-miR-6733-5p
74311,rs6605067,chr1:944296,A,ENSG00000188976,ENST00000870726,Transcript,3_prime_UTR_variant,2768,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1,True,hsa-miR-205-3p;hsa-miR-6733-5p
74312,rs6605067,chr1:944296,A,ENSG00000188976,ENST00000870727,Transcript,3_prime_UTR_variant,2684,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1,True,hsa-miR-205-3p;hsa-miR-6733-5p


In [ ]:
def findDisrupted(row):
    pos = row['cDNA_position']
    local_dftarp = dftarp[dftarp['mRNA'] == row['Feature']]
    if local_dftarp.empty:
        return None

    if '-' in pos:
        pos1, pos2 = map(int, pos.split('-', 1))
    else:
        pos1 = pos2 = int(pos)

    hits = pd.concat([
        local_dftarp[
            (local_dftarp['bs_start'] <= pos_iter) &
            (local_dftarp['bs_end'] >= pos_iter)
        ]
        for pos_iter in range(pos1, pos2 + 1)
    ], ignore_index=True).drop_duplicates()
    
    if hits.empty:
        return None

    return ';'.join(sorted(hits['miRNA'].unique()))


df2.loc[:, 'miRNADisrupt'] = df2.apply(findDisrupted, axis=1)

/home/nikos/miniconda3/envs/TarPmiR/lib/python3.7/site-packages/pandas/core/indexing.py:1667: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[key] = value


In [5]:
dftarp = dftarp[['miRNA', 'mRNA']].join(dftarp['binding_site'].str.split(',', expand=True).astype(int).rename(columns={0: 'bs_start', 1: 'bs_end'}))

dftarp

,miRNA,mRNA,bs_start,bs_end
0,hsa-let-7a-5p,ENST00000341065,1033,1047
1,hsa-let-7a-5p,ENST00000341065,117,133
2,hsa-let-7a-3p,ENST00000341065,166,187
3,hsa-let-7a-2-3p,ENST00000341065,1378,1402
4,hsa-let-7a-2-3p,ENST00000341065,1933,1961
...,...,...,...,...
35049,hsa-miR-3059-3p,ENST00000870737,1956,1982
35050,hsa-miR-3085-5p,ENST00000870737,920,981
35051,hsa-miR-3085-3p,ENST00000870737,1924,1949
35052,hsa-miR-3085-3p,ENST00000870737,1829,1849


In [33]:
dftarp[dftarp['mRNA'] == 'ENST00000641515']

,miRNA,mRNA,bs_start,bs_end
0,hsa-let-7a-5p,ENST00000641515,1786,1811
1,hsa-let-7a-2-3p,ENST00000641515,646,674
2,hsa-let-7b-5p,ENST00000641515,1140,1188
3,hsa-let-7b-5p,ENST00000641515,274,301
4,hsa-let-7b-3p,ENST00000641515,2483,2524
...,...,...,...,...
1445,hsa-miR-3059-3p,ENST00000641515,1618,1636
1446,hsa-miR-3085-5p,ENST00000641515,1651,1707
1447,hsa-miR-3085-5p,ENST00000641515,1620,1648
1448,hsa-miR-3085-3p,ENST00000641515,547,580


In [17]:
df2[['Feature', 'cDNA_position', 'miRNADisrupt']]

,Feature,cDNA_position,miRNADisrupt
11793,ENST00000641515,2331,hsa-miR-224-5p;hsa-miR-3126-3p;hsa-miR-5586-3p
74226,ENST00000870725,3483-3485,None
74292,ENST00000327044,2664,hsa-miR-205-3p;hsa-miR-6733-5p
74293,ENST00000341065,1912,hsa-miR-152-5p;hsa-miR-301a-5p;hsa-miR-550a-3p...
74294,ENST00000342066,2279,hsa-miR-152-5p;hsa-miR-550a-3p;hsa-miR-627-3p


In [11]:
import requests
import re

transcriptsNum = 100

df2 = df.head(transcriptsNum)
downloadTranscripts = df2['Feature'].drop_duplicates().to_list()

url = "http://www.ensembl.org/biomart/martservice"
query_xml = f"""<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE Query>
<Query  virtualSchemaName = "default" formatter = "FASTA" header = "0" uniqueRows = "0" count = "" datasetConfigVersion = "0.6" >
			
	<Dataset name = "hsapiens_gene_ensembl" interface = "default" >
		<Filter name = "ensembl_transcript_id" value = "{','.join(downloadTranscripts)}"/>
		<Attribute name = "ensembl_transcript_id" />
		<Attribute name = "cdna" />
	</Dataset>
</Query>"""

res = requests.get(url, params={"query": query_xml})
with open('./TarPmiR_Linux/cDNA.fasta', 'w') as file:
    file.write(res.text)
    
with open('./TarPmiR_Linux/cDNA.fasta', 'r', encoding='utf-8') as f:
    fasta_ids = [line.strip()[1:] for line in f if line.startswith('>')]
    
testTarPmiR()
    
with open('./TarPmiR_Linux/mirnaverified.fasta_cDNA.fasta.bp', 'r', encoding='utf-8') as f_in:
    content = f_in.read()

content = re.sub(r'(?<=.)hsa', r'\nhsa', content)

with open('./TarPmiR_Linux/mirnaverified.fasta_cDNA.fasta.bp', 'w', encoding='utf-8') as f_out:
    f_out.write(content)
dftarp = pd.read_csv(
    './TarPmiR_Linux/mirnaverified.fasta_cDNA.fasta.bp',
    sep='\t'
)
dftarp.columns = ['miRNA','mRNA','binding_site','binding_probability','energy','seed','accessibility','AU_content','PhyloP_Stem','PyloP_Flanking','m/e','number_of_pairings','binding_region_length','longest_consecutive_pairings','position_of_longest_consecutive_pairings','pairings_in_3prime_end','difference_of_pairings_between_seed_and_3prime_end']
dftarp.loc[:, 'miRNA'] = dftarp['miRNA'].str.split().str[0]
binds = dftarp.groupby('mRNA')['miRNA'].apply(lambda s: ';'.join(map(str, s))).to_dict()

df2.loc[:, 'hasSequence'] = df2['Feature'].isin(fasta_ids)
df2.loc[:, 'miRNAHits'] = df2['Feature'].map(binds).where(df2['Feature'].map(binds).notna(), None)

df2

/home/nikos/miniconda3/envs/TarPmiR/lib/python3.7/site-packages/pandas/core/indexing.py:1667: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[key] = value


,Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,Existing_variation,Extra,hasSequence,miRNAHits
11793,chr1_71298_G/A,chr1:71298,A,ENSG00000186092,ENST00000641515,Transcript,3_prime_UTR_variant,2331,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1,True,hsa-let-7a-5p;hsa-let-7b-5p;hsa-let-7c-5p;hsa-...
74226,rs56972694,chr1:943527-943529,-,ENSG00000188976,ENST00000870725,Transcript,3_prime_UTR_variant,3483-3485,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1,True,hsa-let-7b-5p;hsa-let-7c-5p;hsa-miR-15a-5p;hsa...
74292,rs6605067,chr1:944296,A,ENSG00000188976,ENST00000327044,Transcript,3_prime_UTR_variant,2664,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1,True,hsa-let-7a-5p;hsa-let-7a-2-3p;hsa-let-7b-5p;hs...
74293,rs6605067,chr1:944296,A,ENSG00000187634,ENST00000341065,Transcript,3_prime_UTR_variant,1912,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1;FLAGS=cds_start_NF,True,hsa-let-7a-5p;hsa-let-7b-5p;hsa-let-7c-5p;hsa-...
74294,rs6605067,chr1:944296,A,ENSG00000187634,ENST00000342066,Transcript,3_prime_UTR_variant,2279,-,-,-,-,-,IMPACT=MODIFIER;STRAND=1,True,hsa-let-7e-3p;hsa-miR-17-3p;hsa-miR-18a-5p;hsa...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74411,rs2839,chr1:944307,C,ENSG00000188976,ENST00000934934,Transcript,3_prime_UTR_variant,2695,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1,True,hsa-let-7a-2-3p;hsa-let-7e-3p;hsa-miR-15a-5p;h...
74412,rs2839,chr1:944307,C,ENSG00000188976,ENST00000934935,Transcript,3_prime_UTR_variant,2634,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1,True,hsa-let-7a-2-3p;hsa-let-7e-3p;hsa-miR-15a-5p;h...
74413,rs2839,chr1:944307,C,ENSG00000188976,ENST00000934936,Transcript,3_prime_UTR_variant,2424,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1,True,hsa-let-7a-2-3p;hsa-let-7e-3p;hsa-miR-15a-5p;h...
74414,rs2839,chr1:944307,C,ENSG00000188976,ENST00000934937,Transcript,3_prime_UTR_variant,2684,-,-,-,-,-,IMPACT=MODIFIER;STRAND=-1,True,hsa-let-7a-2-3p;hsa-let-7b-5p;hsa-let-7e-3p;hs...


## ~~Kano requests sto Ensembl me ta transcript ids tou gia na vro ta NM_ id pou einai referenced~~
[Ensembl REST API](https://rest.ensembl.org/)

[API CrossRef GET Request](https://rest.ensembl.org/documentation/info/xref_id)

In [ ]:
import requests

In [ ]:
transcriptid ='ENST00000327044'
res = requests.get(f'https://rest.ensembl.org/xrefs/id/{transcriptid}?external_db=RefSeq_mRNA', headers={"content-type":"application/json"})

res.json()

### Get request - pairnei peripou 0,29 defterolepta gia kathe id (an kano olo to output me 33000 ids tha parei peripou 3 ores)

In [ ]:
dict = {}
for id in transcripts:
    print("Getting " + id)
    res = requests.get(f'https://rest.ensembl.org/xrefs/id/{id}?external_db=RefSeq_mRNA', headers={"content-type":"application/json"}).json()
    if res == []:
        dict[id] = None
        continue
    dict[id] = res[0]['primary_id']
    
dict

### Post request - den fainetai na mporo na paro to NM_ apo ta post requests pou mporo na kano

In [ ]:
idList = ['ENST00000327044', 'ENST00000379370']
server = "https://rest.ensembl.org"
ext = "/lookup/id"
headers={ "Content-Type" : "application/json", "Accept" : "application/json"}
r = requests.post('https://rest.ensembl.org/lookup/id', headers=headers, data='{"ids":["ENST00000327044","ENST00000379370"]}')

r.json()

## TarPmiR test

In [17]:
idList = transcripts[10:15]
res = ''
for transcriptid in idList:
    res += requests.get(f'https://rest.ensembl.org/sequence/id/{transcriptid}?type=cdna', headers={"content-type":"text/x-fasta"}).text

res

'>ENST00000870729.1\nGGGGCCGGAAGTGGGGTGCACGCTTCGGGTTGGTGTCATGGCAGCTGCGGGGAGCCGCAA\nGAGGCGCCTGGCGGAGCTGACGGTGGACGAGTTCCTAGCTTCGGGCTTTGACTCCGAGTC\nCGAATCCGAGTCCGAAAATTCTCCACAAGCGGAGACACGGGAAGCACGCGAGGCTGCCCG\nGAGTCCGGATAAGCCGGGCGGGAGCCCCTCGGCCAGCCGGCGTAAAGGCCGTGCCTCTGA\nGCACAAAGACCAGCTCTCTCGGCTGAAGGACAGAGACCCCGAGTTCTACAAGTTCCTGCA\nGGAGAATGACCAGAGCCTGCTAAACTTCAGCGACTCGGACAGCTCTGAGGAGGAAGAGGG\nGCCGTTCCACTCCCTGCCAGATGTGCTGGAGGAAGCCAGTGAGGAGGAGGATGGAGCGGA\nGGAAGGAGAAGATGGGGACAGAGTCCCCAGAGGGCTGAAGGGGAAGAAGAATTCTGTTCC\nTGTGACCGTCGCCATGGTTGAGAGATGGAAGCAGGCAGCAAAGCAACGCCTCACTCCAAA\nGCTGTTCCATGAAGTGGTACAGGCGTTCCGAGCAGCTGTGGCCACCACCCGAGGGGACCA\nGGAAAGTGCTGAGGCCAACAAATTCCAGGTCACGGACAGTGCTGCATTCAATGCTCTGGT\nTACCTTCTGCATCAGAGACCTCATTGGCTGTCTCCAGAAGCTGCTGTTTGGAAAGGTGGC\nAAAGGATAGCAGCAGCAAATGTGGGCCAAAGGAAACCAGTCTGCGGCAGTTGGTGGTGCC\nTGGGCGACCCCAGAGGATGCTGCAGCCGTCCAGCAGCCCGCTCTGGGGGAAGCTTCGTGT\nGGACATCAAGGCTTACCTGGGCTCGGCCATACAGCTGGTGTCCTGTCTGTCGGAGACGAC\nGGTGTTGGCGGCCGTGCTGCGGCACATCAGCGTGCTGGTGCCCTGCTTC

In [ ]:
import subprocess
import os
import tempfile

with tempfile.NamedTemporaryFile(mode='w', suffix='.fasta', delete=False, encoding='utf-8') as f:
    f.write(res)
    mrna_path = f.name

try:
    result = subprocess.run(
        [
            'python',
            './TarPmiR.py',
            '-a', './mirnaverified.fasta',
            '-b', mrna_path,
            '-m', './models/Human.pkl',
            '-p', '0.01'
        ],
        text=True,
        cwd='/home/nikos/ptixiaki/TarPmiR_Linux'
    )
finally:
    os.unlink(mrna_path)

In [4]:
import subprocess
import os
import requests
import tempfile
import logging
import re
from multiprocessing import Pool

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='run.log',
    filemode='a'
)

def runTarpmir(idList):
    logging.info(f'PID{os.getpid()}: Starting')
    
    res = ''    
    for transcriptid in idList:
        res += requests.get(f'https://rest.ensembl.org/sequence/id/{transcriptid}?type=cdna', headers={"content-type":"text/x-fasta"}).text
    res = re.sub(r'(?m)^You.+connections', '', res) #NOTE: PREPEI NA FTIAKSO TO RATE LIMIT KAPOS
        
    logging.info(f'PID{os.getpid()}: Transcript GET completed')
        
    with tempfile.NamedTemporaryFile(mode='w', suffix='.fasta', delete=False, encoding='utf-8') as f:
        f.write(res)
        mrna_path = f.name


    try:
        logging.info(f'PID{os.getpid()}: Starting TarPmiR with tempfile {mrna_path}')
        result = subprocess.run(
            [
                'python',
                './TarPmiR.py',
                '-a', './mirnaverified.fasta',
                '-b', mrna_path,
                '-m', './models/Human.pkl',
                '-p', '0.01'
            ],
            text=True,
            cwd='/home/nikos/ptixiaki/TarPmiR_Linux',
            capture_output=True,
            check=True
        )
        logging.info(f'PID{os.getpid()}: TarPmiR completed')
    except subprocess.CalledProcessError as e:
        logging.error(f'PID{os.getpid()}: TarPmiR failed with return code {e.returncode} using tempfile {mrna_path}:\n{e.stderr}')
    except Exception as e:
        logging.error(f'PID{os.getpid()}: Exception occurred: {e}')
    # finally:
    #     os.unlink(mrna_path)
        

def chunkify(idList, chunkNum):
    chunk_size = len(idList) // chunkNum
    chunks = [idList[i*chunk_size:(i+1)*chunk_size] for i in range(chunkNum)]

    if len(idList) % chunkNum != 0:
        chunks[-1].extend(idList[chunkNum*chunk_size:])
    return chunks


def batchTarPmiR(idList, processNum):
    logging.info(f'STARTING BATCH PROCESS WITH {processNum} PROCESSES')
    with Pool(processes=processNum) as pool:
        pool.map(runTarpmir, chunkify(idList, processNum))
    logging.info(f'FINISHED BATCH PROCESS')

In [5]:
batchTarPmiR(transcripts[:5000], 22)

In [9]:
def testTarPmiR():
    try:
        result = subprocess.run(
            [
                'python',
                './TarPmiR.py',
                '-a', './mirnaverified.fasta',
                '-b', './cDNA.fasta',
                '-m', './models/Human_sklearn_0.22.pkl',
                '-p', '0.5'
            ],
            text=True,
            cwd='/home/nikos/ptixiaki/TarPmiR_Linux',
            capture_output=True,
            check=True
        )
        logging.info(f'PID{os.getpid()}: TarPmiR completed')
    except subprocess.CalledProcessError as e:
        logging.error(f'PID{os.getpid()}: TarPmiR failed with return code {e.returncode}:\n{e.stderr}')
    except Exception as e:
        logging.error(f'PID{os.getpid()}: Exception occurred: {e}')